In [1]:
import pandas as pd
import numpy as np
from LLM import Clasificador
from tqdm import tqdm

c:\Users\chris\Data_science\Proyectos_perso\LLMzCor.github.io\env_PAC\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [62]:
file_path = r"C:\Users\chris\Data_science\Proyectos_perso\LLMzCor.github.io\DBs\Clostridium_difficile.xlsx"
sheet = "test"
usecols= [
    'PMID',
    'Title',
    'Abstract',
    '1) Antimicrobial Resistance stain',
    '2) New treatment',
    '3) Immunization',
    'Human_summary'
]
df = pd.read_excel(
    io=file_path,
    sheet_name=sheet,
    usecols=usecols,
    index_col=0
)

In [67]:
clasificador=Clasificador()

In [10]:
f=df["2) New treatment"] == "No"
df.loc[f,"2) New treatment"]=0

In [11]:
f=df["2) New treatment"] == "Yes"
df.loc[f,"2) New treatment"]=1

In [13]:
f=df["1) Antimicrobial Resistance stain"] == "No"
df.loc[f,"1) Antimicrobial Resistance stain"]=0

In [14]:
f=df["1) Antimicrobial Resistance stain"] == "Yes"
df.loc[f,"1) Antimicrobial Resistance stain"]=1

In [68]:
df

,Title,Abstract,1) Antimicrobial Resistance stain,2) New treatment,3) Immunization,Human_summary
PMID,,,,,,
36466927,Neutralizing epitopes on Clostridioides diffic...,Toxin A (TcdA) and toxin B (TcdB) are two key ...,No,Yes,No,Using antibodies (VHHs) AH3 and AA6 are two po...
36439832,Peroxisome proliferator-activated receptor-γ a...,Clostridioides difficile is a major causative ...,No,Yes,No,"Administration of the PPAR-γ agonist, pioglita..."
36439215,The impact of dietary fibers on Clostridioides...,Diets rich in fiber may provide health benefit...,No,Yes,No,Use Inulin or pectin as a dietary-based therap...
36312948,Receptor binding protein of prophage reversibl...,Receptor-binding proteins (RBPs) are located a...,No,Yes,No,The paper studied a protein named PtsHN10M tha...
35042668,The emergence of Clostridioides difficile PCR ...,Background: Several studies have highlighted t...,Yes,No,No,This study focused on analyzing Clostridioides...
36093337,Antibiotic resistance and genomic features of ...,Background: Clostridioides difficile infection...,Yes,No,No,While current main treatments remain effective...
36026500,Characterization of the virulence of three nov...,Clostridioides (Clostridium) difficile infecti...,Yes,No,No,This study explored the emergence of novel Clo...
35939437,Discovery of a novel natural product inhibitor...,Clostridioides (Clostridium) difficile infecti...,Yes,No,No,This study was conducted in Brazil to better u...
33958010,Associations of facility-level antibiotic use ...,Previously reported associations between hospi...,Yes,No,No,This study reanalyzed the link between hospita...


In [69]:
esponse = clasificador.clasificacion(df.loc[34496836,"Abstract"])

KeyboardInterrupt: 

In [64]:
def _transformacion_binario_val(col):
    if (col=="Yes") | (col==1):
        return 1
    elif (col=="No") | (col==0):
        return 0
def _transformacion_binario_df(df):
    df[['1) Antimicrobial Resistance stain',
        '2) New treatment','3) Immunization']] = df[['1) Antimicrobial Resistance stain',
                                                     '2) New treatment',
                                                     '3) Immunization']].applymap(_transformacion_binario_val)
    return df



def ask_llm(df_,partition=0):
    df=df_.copy()
    df=_transformacion_binario_df(df)
    if partition==0:
        for pmid in df.index:
            response = clasificador.clasificacion(df.loc[pmid,"Abstract"])
            df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
        return df
    else:
        print("aca")
        df_ptit=df.iloc[:10,:]
        print(df_ptit.index)
        for pmid in df_ptit.index:
            response = clasificador.clasificacion(df.loc[pmid,"Abstract"])
            df.loc[pmid,["ai_label","ai_summary"]] = response[0] , response[1]
        return df_ptit

def _evaluate(row):
    values = row[["1) Antimicrobial Resistance stain","2) New treatment","3) Immunization"]].values
    ai_opcion=int(row["ai_label"])-1
    print(f"values={values}")
    print(f"ai_opcion{ai_opcion}")
    if (len(values)==0) & (ai_opcion==3):
        return 1
    elif (len(values)==0) & (ai_opcion<3):
        return 0
    elif (len(values)>0) & (ai_opcion==3):
        return 0
    elif values[ai_opcion]>0:

        return 1
    else:
        return 0

def evaluacion_score(df,partition=0):
    if partition ==0:
        n=df.shape[0]
        scores = df.apply(_evaluate,axis=1)
        final_score=sum(scores)/n
    else:
        df_ptit=df.iloc[:10,:]
        n=df_ptit.shape[0]
        scores=df_ptit.apply(_evaluate,axis=1)
        final_score=sum(scores)/n
    return final_score

In [65]:
df

,Title,Abstract,1) Antimicrobial Resistance stain,2) New treatment,3) Immunization,Human_summary
PMID,,,,,,
36466927,Neutralizing epitopes on Clostridioides diffic...,Toxin A (TcdA) and toxin B (TcdB) are two key ...,No,Yes,No,Using antibodies (VHHs) AH3 and AA6 are two po...
36439832,Peroxisome proliferator-activated receptor-γ a...,Clostridioides difficile is a major causative ...,No,Yes,No,"Administration of the PPAR-γ agonist, pioglita..."
36439215,The impact of dietary fibers on Clostridioides...,Diets rich in fiber may provide health benefit...,No,Yes,No,Use Inulin or pectin as a dietary-based therap...
36312948,Receptor binding protein of prophage reversibl...,Receptor-binding proteins (RBPs) are located a...,No,Yes,No,The paper studied a protein named PtsHN10M tha...
35042668,The emergence of Clostridioides difficile PCR ...,Background: Several studies have highlighted t...,Yes,No,No,This study focused on analyzing Clostridioides...
36093337,Antibiotic resistance and genomic features of ...,Background: Clostridioides difficile infection...,Yes,No,No,While current main treatments remain effective...
36026500,Characterization of the virulence of three nov...,Clostridioides (Clostridium) difficile infecti...,Yes,No,No,This study explored the emergence of novel Clo...
35939437,Discovery of a novel natural product inhibitor...,Clostridioides (Clostridium) difficile infecti...,Yes,No,No,This study was conducted in Brazil to better u...
33958010,Associations of facility-level antibiotic use ...,Previously reported associations between hospi...,Yes,No,No,This study reanalyzed the link between hospita...


In [66]:
df_new=ask_llm(df)

C:\Users\chris\AppData\Local\Temp\ipykernel_29648\3939741552.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  '3) Immunization']].applymap(_transformacion_binario_val)


InferenceTimeoutError: Model not loaded on the server: https://api-inference.huggingface.co/models/mistralai/Mixtral-8x7B-Instruct-v0.1. Please retry with a higher timeout (current: 120).

In [59]:
df_new

,Title,Abstract,1) Antimicrobial Resistance stain,2) New treatment,3) Immunization,Human_summary,ai_label,ai_summary
PMID,,,,,,,,
36466927,Neutralizing epitopes on Clostridioides diffic...,Toxin A (TcdA) and toxin B (TcdB) are two key ...,0,1,0,Using antibodies (VHHs) AH3 and AA6 are two po...,3,"""The paper discusses the use of single-domain..."
36439832,Peroxisome proliferator-activated receptor-γ a...,Clostridioides difficile is a major causative ...,0,1,0,"Administration of the PPAR-γ agonist, pioglita...",NaN,NaN
36439215,The impact of dietary fibers on Clostridioides...,Diets rich in fiber may provide health benefit...,0,1,0,Use Inulin or pectin as a dietary-based therap...,NaN,NaN
36312948,Receptor binding protein of prophage reversibl...,Receptor-binding proteins (RBPs) are located a...,0,1,0,The paper studied a protein named PtsHN10M tha...,NaN,NaN
35042668,The emergence of Clostridioides difficile PCR ...,Background: Several studies have highlighted t...,1,0,0,This study focused on analyzing Clostridioides...,NaN,NaN
36093337,Antibiotic resistance and genomic features of ...,Background: Clostridioides difficile infection...,1,0,0,While current main treatments remain effective...,NaN,NaN
36026500,Characterization of the virulence of three nov...,Clostridioides (Clostridium) difficile infecti...,1,0,0,This study explored the emergence of novel Clo...,NaN,NaN
35939437,Discovery of a novel natural product inhibitor...,Clostridioides (Clostridium) difficile infecti...,1,0,0,This study was conducted in Brazil to better u...,NaN,NaN
33958010,Associations of facility-level antibiotic use ...,Previously reported associations between hospi...,1,0,0,This study reanalyzed the link between hospita...,NaN,NaN
